In [ ]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [ ]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "Rainbow File"))


In [ ]:
from src.demos.ekg.pydantic_rag import PydanticRag

KV_STORE = "file"
KV_STORE = None

url = global_config().get_dsn("vector_store.postgres_url", driver="asyncpg")

rag = PydanticRag(
    model_definition=test_schema, postgres_url=url, embeddings_id="qwen3_06b_deepinfra", llm_id=None, kv_store_id=KV_STORE
)


In [ ]:
import os
from pathlib import Path

doc_id = "03.RESM-SOL-9000559500_CNES_TMA_VENUS_VIP_PEPS_THEIA_MUSCATE-v0.2_extracted.json"

file1 = Path(os.getenv("ONEDRIVE", "")) / "prj/atos-kg/rainbow-json/" / doc_id
assert file1.exists()
doc_text = file1.read_text()

rainbow_report = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", rainbow_report)

assert rainbow_report


In [ ]:
from src.utils.pydantic.kv_store import save_object_to_kvstore


save_object_to_kvstore(doc_id,rainbow_report)

In [ ]:
debug(rainbow_report)

In [ ]:
chunks = rag.chunck(rainbow_report)
debug(chunks)


In [ ]:
# 2. Index the document
# rag.store_chunks(chunks)
print("Document stored.")


In [ ]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

In [ ]:
# 3. Query the vector store
hits = rag.query_vectorstore(
    "e-mail address", k=2, filter={"langchain_metadata ->> description": {"$eq": "Age in years"}}
)
print("Vector hits:", hits)